# NB0 — Harvesting a raw Arabic news corpus from CulturaX

My previous human corpus turned out to be **sanitized**: an earlier cleaning pass had stripped
digits, quotation marks, colons and most punctuation. That's fatal for this thesis — a detector
could separate the classes on "has a digit → AI" with zero linguistic understanding. So I'm
rebuilding the human side from CulturaX, but this time I keep the text **raw** (no symbol
stripping at all), and I select for the genre my thesis targets: political / Middle-East / hard
news from trusted non-Western Arabic outlets.

**Strategy:** CulturaX Arabic is tens of GB, so I **stream** it (never download the whole thing)
and keep only documents that pass three filters — source domain, topic keywords, and length —
until I have 10k–15k articles.

**Output:** `culturax_raw_harvest.parquet` — the pool that NB0b will rank down to the best 3,500.

**Critical rule:** I do NOT clean, normalize, or strip anything here. Raw text in, raw text out.
The only transformation is whitespace trimming. Keeping digits/punctuation is the whole point.

## Config — sources, topic keywords, length, target

In [1]:
import re, json, os, time
import pandas as pd
from datasets import load_dataset

# --- trusted non-Western Arabic outlets (domain substrings matched against each doc's url) ---
SOURCE_DOMAINS = [
    'aljazeera.net', 'aljazeera.com',
    'alarabiya.net',
    'almayadeen.net',
    'arabic.rt.com',
    'aa.com.tr',            # Anadolu (Arabic section)
    'arabi21.com',
    'alquds.co.uk',
    'alaraby.co.uk', 'alaraby.tv',   # التلفزيون العربي / العربي الجديد
    'maannews.net',        # Ma'an
    'wafa.ps',             # WAFA
    'qudspress.com',
]

# --- topic filter: political / Middle-East / Palestine hard news ---
TOPIC_KEYWORDS = [
    'فلسطين','غزة','الضفة','القدس','الاحتلال','حماس','السلطة الفلسطينية','منظمة التحرير',
    'إسرائيل','الاحتلال الإسرائيلي','المقاومة','الانتفاضة','رام الله','حزب الله','لبنان',
    'سوريا','العراق','اليمن','إيران','مصر','الأردن','السعودية','قطر','تركيا',
    'الشرق الأوسط','مجلس الأمن','الأمم المتحدة','وزارة الخارجية','الرئيس','الحكومة',
    'انتخابات','مفاوضات','اتفاق','قمة','عقوبات','الجيش','غارة','قصف','هدنة','وقف إطلاق النار',
    'الوزراء','البرلمان','مظاهرات','احتجاجات','أزمة','صراع','نزاع','عملية عسكرية',
]

MIN_WORDS   = 400          # my thesis scope: long-form reports
MAX_WORDS   = 6000         # guard against scraped junk / concatenated pages
TARGET      = 15000        # harvest ceiling (I'll rank down to 3,500 in NB0b)
MIN_KW_HITS = 2            # a doc must contain at least this many topic keywords

OUT_DIR     = '/kaggle/working'
CKPT        = f'{OUT_DIR}/culturax_raw_harvest.parquet'
SAVE_EVERY  = 100
print('config ready | target', TARGET, '| sources', len(SOURCE_DOMAINS))

config ready | target 15000 | sources 13


## Filters (raw-preserving)

`domain_ok` matches the document URL against my source list. `topic_ok` requires at least
`MIN_KW_HITS` distinct topic keywords so I get substantive political coverage, not a passing
mention. `length_ok` enforces the long-form range. None of these modify the text.

In [2]:
def domain_ok(url):
    if not url:
        return None
    u = url.lower()
    for d in SOURCE_DOMAINS:
        if d in u:
            return d
    return None

def topic_hits(text):
    return sum(1 for kw in TOPIC_KEYWORDS if kw in text)

def length_ok(text):
    n = len(text.split())
    return MIN_WORDS <= n <= MAX_WORDS, n

def raw_trim(text):
    # the ONLY transformation allowed here: collapse whitespace. No symbol stripping.
    return re.sub(r'[ \t]+', ' ', str(text)).strip()

## Stream CulturaX Arabic and harvest

I open the Arabic config in streaming mode and iterate. For every doc I check domain → topic →
length, keep the ones that pass, and print a live line so I can watch progress and stop early if
needed. I checkpoint every 100 kept articles, and I can resume across Save Version runs.

If CulturaX needs authentication, I attach my HF token as the Kaggle secret `HF_TOKEN`.

In [3]:
# HF auth (CulturaX requires accepting terms; attach HF_TOKEN as a Kaggle secret)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token attached')
except Exception as e:
    print('No HF_TOKEN secret found — trying anonymous access:', str(e)[:80])

# resume if a checkpoint already exists
kept, seen_urls = [], set()
if os.path.exists(CKPT):
    prev = pd.read_parquet(CKPT)
    kept = prev.to_dict('records')
    seen_urls = set(prev['url'])
    print(f'resuming: {len(kept)} already harvested')

ds = load_dataset('uonlp/CulturaX', 'ar', split='train', streaming=True)
print('stream opened', flush=True)

HF token attached


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]

stream opened


In [4]:
t0 = time.time()
scanned = 0
by_source = {}

for doc in ds:
    if len(kept) >= TARGET:
        break
    scanned += 1

    # live heartbeat every 2000 scanned docs so I know it's alive even when few pass
    if scanned % 2000 == 0:
        rate = scanned / max(time.time() - t0, 1)
        print(f'  scanned {scanned:,} | kept {len(kept)} | {rate:.0f} docs/s | '
              f'sources {by_source}', flush=True)

    url = doc.get('url', '') or ''
    src = domain_ok(url)
    if not src:
        continue
    if url in seen_urls:
        continue

    text = raw_trim(doc.get('text', ''))
    hits = topic_hits(text)
    if hits < MIN_KW_HITS:
        continue
    lok, nwords = length_ok(text)
    if not lok:
        continue

    # keep it
    seen_urls.add(url)
    by_source[src] = by_source.get(src, 0) + 1
    kept.append({'text': text, 'url': url, 'source_domain': src,
                 'timestamp': doc.get('timestamp', ''), 'n_words': nwords,
                 'topic_hits': hits})

    # print EVERY kept article (per request) so I can follow and stop when needed
    print(f'[{len(kept):5d}/{TARGET}] {src:16s} | {nwords:5d}w | kw={hits:2d} | '
          f'{text[:60]}...', flush=True)

    if len(kept) % SAVE_EVERY == 0:
        pd.DataFrame(kept).to_parquet(CKPT, index=False)
        print(f'    >> checkpoint saved: {len(kept)} articles | scanned {scanned:,}', flush=True)

# final save
pd.DataFrame(kept).to_parquet(CKPT, index=False)
print(f'\nDONE | kept {len(kept)} | scanned {scanned:,} | {time.time()-t0:.0f}s', flush=True)
print('by source:', by_source, flush=True)

[    1/15000] aljazeera.net    |   548w | kw= 9 | ثلاث قوائم تتكتل لمفاوضات تشكيل الحكومة العراقية المقبلة
الس...
[    2/15000] aljazeera.net    |   655w | kw= 8 | السعودية تقصف مواقع للحوثيين البث الحي
الخميس 18/11/1430 هـ ...
[    3/15000] aljazeera.net    |   606w | kw= 4 | الأحد 1435/3/10 هـ - الموافق 12/1/2014 م (آخر تحديث) الساعة ...
[    4/15000] aljazeera.net    |  1622w | kw=21 | بعد حفلات التطبيع.. هل يمهِّد الاتفاقُ طريق دحلان نحو رام ال...
[    5/15000] aljazeera.net    |   520w | kw= 6 | فقدت الليرة السورية نحو ربع قيمتها هذا الشهر أمام الدولار، و...
[    6/15000] alaraby.co.uk    |   644w | kw= 3 | ولاية "شليسفيغ هولشتاين" الألمانية تنتخب نوابها
يتوجه الناخب...
[    7/15000] alarabiya.net    |   533w | kw= 3 | وزارة الإسكان تشعبت عن أهدافها
نشر في: 19 فبراير ,2016: 12:0...
[    8/15000] alarabiya.net    |   473w | kw= 7 | آخر تحديث: الأحد 29 ربيع الأول 1437 هـ - 10 يناير 2016 KSA 1...
[    9/15000] arabic.rt.com    |   462w | kw= 3 | يوم الغضب في سوريا بين مؤيد ومعارض - R

## Verify the harvest is RAW (the whole point)

I confirm the harvested text actually contains digits, quotation marks, colons and parentheses —
the symbols that were missing from my old sanitized corpus. If these are present, the source
problem is solved.

In [5]:
h = pd.read_parquet(CKPT)
print('harvested:', h.shape)
print('by source:\n', h['source_domain'].value_counts())
print('\nlength: mean %.0f | median %.0f | min %d | max %d' %
      (h['n_words'].mean(), h['n_words'].median(), h['n_words'].min(), h['n_words'].max()))

import numpy as np
def frac(pat): return np.mean([bool(re.search(pat, t)) for t in h['text']])
print('\n--- RAW check (these were ~0% in the old sanitized corpus) ---')
print('has digits      :', f'{frac(chr(91)+"0-9٠-٩"+chr(93)):.0%}')
print('has quotes      :', f'{frac(chr(34)+"|"+chr(171)+"|"+chr(187)):.0%}')
print('has colon       :', f'{frac(":"):.0%}')
print('has parentheses :', f'{frac(chr(92)+"("):.0%}')
print('mean digits/article:', f"{np.mean([len(re.findall(chr(91)+'0-9٠-٩'+chr(93), t)) for t in h['text']]):.1f}")

print('\n--- sample article ---')
print(h.iloc[0]['text'][:500])

harvested: (15000, 6)
by source:
 source_domain
aljazeera.net     7903
alaraby.co.uk     2731
arabi21.com       1259
alarabiya.net     1117
arabic.rt.com      670
almayadeen.net     470
maannews.net       341
alquds.co.uk       217
qudspress.com      182
aa.com.tr           52
wafa.ps             42
aljazeera.com        8
alaraby.tv           8
Name: count, dtype: int64

length: mean 806 | median 596 | min 400 | max 5969

--- RAW check (these were ~0% in the old sanitized corpus) ---
has digits      : 95%
has quotes      : 91%
has colon       : 80%
has parentheses : 82%
mean digits/article: 38.1

--- sample article ---
ثلاث قوائم تتكتل لمفاوضات تشكيل الحكومة العراقية المقبلة
السبت 1426/12/27 هـ - الموافق 28/1/2006 م (آخر تحديث) الساعة 8:10 (مكة المكرمة)، 5:10 (غرينتش)
قوى سياسية عراقية سنية تتكتل مع قائمة علاوي لبحث تشكيل حكومة وحدة وطنية (الفرنسية)
اتفقت القائمة الوطنية العراقية بزعامة رئيس الوزراء السابق إياد علاوي وجبهة التوافق الوطنية (السنية) والجبهة العراقية للحوار الوطني، على ال

## Notes

- **No sanitization here on purpose.** The only text change is whitespace trimming; digits,
  punctuation, quotes and parentheses are all preserved. That's what makes this corpus a valid
  human counterpart to the AI articles.
- `source_domain`, `url`, `timestamp`, `topic_hits` are kept for transparency and for the
  ranking step in NB0b.
- Deduplication here is URL-level only. NB0b does the content-level dedup and picks the final
  3,500 by length distribution, source balance, and topic coverage.
- If the stream is slow or the quota runs low, the checkpoint lets me resume from where I stopped
  in the next Save Version run — I never lose harvested articles.